In [1]:
import joblib

X_train = joblib.load(
    "X_train_random.pkl"
)

X_test = joblib.load(
    "X_test_random.pkl"
)

y_train = joblib.load(
    "y_train_random.pkl"
)

y_test = joblib.load(
    "y_test_random.pkl"
)

In [2]:
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    f_regression
)

In [3]:
preprocessor = Pipeline([

    (
        "imputer",
        SimpleImputer(strategy="median")
    ),

    (
        "variance_filter",
        VarianceThreshold(threshold=0.01)
    ),

    (
        "scaler",
        StandardScaler()
    ),

    (
        "feature_selection",
        SelectKBest(
            score_func=f_regression,
            k=1000
        )
    )

])

In [ ]:
import optuna

from lightgbm import LGBMRegressor

from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate


def objective(trial):

    model = LGBMRegressor(

        n_estimators=trial.suggest_int(
            "n_estimators",
            200,
            1500
        ),

        learning_rate=trial.suggest_float(
            "learning_rate",
            0.01,
            0.3,
            log=True
        ),

        max_depth=trial.suggest_int(
            "max_depth",
            3,
            15
        ),

        num_leaves=trial.suggest_int(
            "num_leaves",
            20,
            300
        ),

        min_child_samples=trial.suggest_int(
            "min_child_samples",
            5,
            100
        ),

        subsample=trial.suggest_float(
            "subsample",
            0.6,
            1.0
        ),

        colsample_bytree=trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0
        ),

        reg_alpha=trial.suggest_float(
            "reg_alpha",
            0,
            10
        ),

        reg_lambda=trial.suggest_float(
            "reg_lambda",
            0,
            10
        ),

        random_state=42,
        n_jobs=-1,
        verbose=-1
    )

    pipeline = Pipeline([

        (
            "preprocessing",
            preprocessor
        ),

        (
            "model",
            model
        )

    ])

    scores = cross_validate(

        pipeline,

        X_train,

        y_train,

        cv=5,

        scoring={"r2": "r2", "mae": "neg_mean_absolute_error"},

        n_jobs=-1

    )

    return scores["test_r2"].mean()

In [5]:
study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=30,
    show_progress_bar=True
)

[I 2026-06-03 11:31:51,865] A new study created in memory with name: no-name-93908601-8ebd-4c1e-bf76-463ac6d9e6ab


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-06-03 11:39:12,779] Trial 0 finished with value: 0.7961756314126365 and parameters: {'n_estimators': 1197, 'learning_rate': 0.012164763966178393, 'max_depth': 14, 'num_leaves': 168, 'min_child_samples': 26, 'subsample': 0.9466505129979353, 'colsample_bytree': 0.7206460102819131, 'reg_alpha': 3.508475103647415, 'reg_lambda': 9.072485147199986}. Best is trial 0 with value: 0.7961756314126365.
[I 2026-06-03 11:39:51,680] Trial 1 finished with value: 0.7854501968157348 and parameters: {'n_estimators': 281, 'learning_rate': 0.020070105793041425, 'max_depth': 6, 'num_leaves': 203, 'min_child_samples': 33, 'subsample': 0.619409060567984, 'colsample_bytree': 0.879680808539445, 'reg_alpha': 4.886633099425789, 'reg_lambda': 6.536873598099593}. Best is trial 0 with value: 0.7961756314126365.
[I 2026-06-03 11:41:31,965] Trial 2 finished with value: 0.7923939363941089 and parameters: {'n_estimators': 1274, 'learning_rate': 0.018549394003695617, 'max_depth': 4, 'num_leaves': 49, 'min_child_s

In [6]:
print("Best CV R2:")
print(study.best_value)

print()

print("Best Parameters:")
print(study.best_params)

Best CV R2:
0.8003596356130929

Best Parameters:
{'n_estimators': 1072, 'learning_rate': 0.027731992707918377, 'max_depth': 12, 'num_leaves': 134, 'min_child_samples': 96, 'subsample': 0.7774776656147349, 'colsample_bytree': 0.779806593248373, 'reg_alpha': 3.8085011447023525, 'reg_lambda': 1.444229223731862}


In [7]:
best_lgbm_pipeline = Pipeline([

    (
        "preprocessing",
        preprocessor
    ),

    (
        "model",
        LGBMRegressor(

            **study.best_params,

            random_state=42,

            n_jobs=-1,

            verbose=-1
        )
    )

])

best_lgbm_pipeline.fit(
    X_train,
    y_train
)

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,steps,"[('imputer', ...), ('variance_filter', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None


In [8]:
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

import numpy as np

y_train_pred = best_lgbm_pipeline.predict(X_train)
y_test_pred = best_lgbm_pipeline.predict(X_test)

print("Train R2 :", r2_score(y_train, y_train_pred))
print("Test R2  :", r2_score(y_test, y_test_pred))

print("Train RMSE :", np.sqrt(mean_squared_error(y_train, y_train_pred)))
print("Test RMSE  :", np.sqrt(mean_squared_error(y_test, y_test_pred)))

print("Train MAE :", mean_absolute_error(y_train, y_train_pred))
print("Test MAE  :", mean_absolute_error(y_test, y_test_pred))

c:\Users\jeshw\anaconda3\envs\solubility_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Train R2 : 0.9469573230362444
Test R2  : 0.8225711037452854
Train RMSE : 0.5476011696601382
Test RMSE  : 0.9807882079637391
Train MAE : 0.2919634730136689
Test MAE  : 0.6369399802617237


c:\Users\jeshw\anaconda3\envs\solubility_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [11]:


print("Best CV R2:")
print(study.best_value)

print()

print("Best Parameters:")
print(study.best_params)

Best CV R2:
0.8003596356130929

Best Parameters:
{'n_estimators': 1072, 'learning_rate': 0.027731992707918377, 'max_depth': 12, 'num_leaves': 134, 'min_child_samples': 96, 'subsample': 0.7774776656147349, 'colsample_bytree': 0.779806593248373, 'reg_alpha': 3.8085011447023525, 'reg_lambda': 1.444229223731862}


NameError: name 'best_svr_pipeline' is not defined